# Encode Real Depositions


In [ ]:
import json, re, glob
from pathlib import Path
from collections import defaultdict, Counter

import torch
import torch.nn.functional as F
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.cm as cm
from scipy.ndimage import gaussian_filter1d
from transformers import AutoTokenizer, AutoModelForCausalLM

import sys
sys.path.append(str(BASE_DIR))
from config import DEFAULT_MODEL, TARGET_LAYER, TOKEN_START, BATCH_SIZE, BASE_DIR

DENOISED_VECTORS  = BASE_DIR / "data/emotion_vectors_denoised.pt"
TRANSCRIPTS_DIR   = BASE_DIR / "data"
RESULTS_DIR       = BASE_DIR / "data/depo_results"
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

# Emotions to highlight in per-deposition arc plots (top ones from validation)
HIGHLIGHT_EMOTIONS = [
    "calm", "sad", "resigned", "suspicious", "satisfied",
    "angry", "warm", "hostile", "confident", "anxious"
]

print(f"Transcripts dir : {TRANSCRIPTS_DIR}")
print(f"Results dir     : {RESULTS_DIR}")


## Load Denoised Emotion Vectors


In [ ]:
vecs_dict = torch.load(DENOISED_VECTORS, map_location="cpu", weights_only=False)
emotions  = sorted(vecs_dict.keys())
mat       = torch.stack([vecs_dict[e] for e in emotions], dim=0).float()
mat       = F.normalize(mat, dim=1)
emotion_to_idx = {e: i for i, e in enumerate(emotions)}

print(f"{len(emotions)} emotion vectors loaded")
print(emotions)


## Load Model


In [ ]:
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Device: {device}")

tokenizer = AutoTokenizer.from_pretrained(DEFAULT_MODEL)
tokenizer.padding_side = "right"
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

model = AutoModelForCausalLM.from_pretrained(
    DEFAULT_MODEL,
    torch_dtype=torch.bfloat16,
    device_map="auto",
)
model.eval()
print("Model loaded.")


## Transcript Parser


In [ ]:
def parse_transcript(path: str) -> list[dict]:
    """
    Parse a deposition transcript into a list of turns.
    Handles three formats:
      Format A: '13 Q.  text'  (line numbers, period after Q/A)
      Format B: '·9· · · ·Q.· text'  (middle-dot padding)
      Format C: 'Q: text' / 'A: text'  (simple colon, no line numbers)
    Returns list of dicts: {role, attorney, text, turn_index}
    """
    with open(path, encoding="utf-8", errors="replace") as f:
        raw = f.read()

    lines = raw.splitlines()

    # Detect format: count lines matching each style in first 60 lines
    fmt_ab = sum(1 for l in lines[:60] if re.match(r"^[\s·]*\d+[\s·]*", l))
    fmt_c  = sum(1 for l in lines[:60] if re.match(r"^[QA]:\s", l))
    is_simple = fmt_c > fmt_ab  # Format C dominates

    cleaned = []
    for line in lines:
        if not is_simple:
            line = re.sub(r"^[\s\u00b7]*\d+[\s\u00b7]*", "", line)
        cleaned.append(line)

    current_attorney = None
    turns = []
    current_role = None
    current_text_lines = []

    by_pattern   = re.compile(
        r"BY\s+(MR\.|MS\.|MRS\.|DR\.)\s*([A-Z][A-Z\s\-']+?):\s*$",
        re.IGNORECASE)
    exam_pattern = re.compile(
        r"(?:CROSS|DIRECT|REDIRECT|RECROSS)?[\s\-]*EXAMINATION\s+BY\s+(MR\.|MS\.|MRS\.|DR\.)\s*([A-Z][A-Z\s\-']+?):\s*$",
        re.IGNORECASE)
    qa_pattern   = re.compile(r"^[·\s]*(Q|A)[\.·:]\s*(.*)", re.IGNORECASE)
    intro_pattern = re.compile(
        r"my name is ([A-Z][a-z]+(?:[\-\s][A-Z][a-z]+)+)",
        re.IGNORECASE)

    def flush(role, attorney, lines):
        text = " ".join(lines).strip()
        text = re.sub(r"\s+", " ", text)
        if text and len(text) > 5:
            turns.append({"role": role, "attorney": attorney, "text": text})

    first_q_seen = False

    for line in cleaned:
        stripped = line.strip()

        if re.match(r"^\(?\d{3}\)?\d{3}-\d{4}", stripped):
            continue
        if re.match(r"^A\s*&\s*A Reporting", stripped, re.IGNORECASE):
            continue
        if re.match(r"^Page \d+$", stripped, re.IGNORECASE):
            continue
        if re.match(r"^Veritext", stripped, re.IGNORECASE):
            continue

        exam_match   = exam_pattern.search(stripped)
        by_match     = by_pattern.search(stripped) if not exam_match else None
        header_match = exam_match or by_match
        if header_match:
            if current_role and current_text_lines:
                flush(current_role, current_attorney, current_text_lines)
                current_text_lines = []
                current_role = None
            title = header_match.group(1).strip()
            name  = header_match.group(2).strip().title()
            current_attorney = f"{title} {name}"
            continue

        qa_match = qa_pattern.match(line)
        if qa_match:
            if current_role and current_text_lines:
                flush(current_role, current_attorney, current_text_lines)
                current_text_lines = []
            current_role = "attorney" if qa_match.group(1).upper() == "Q" else "witness"
            rest = qa_match.group(2).strip()
            current_text_lines = [rest] if rest else []

            if current_role == "attorney" and not first_q_seen:
                first_q_seen = True
                intro_m = intro_pattern.search(rest)
                if intro_m and not current_attorney:
                    current_attorney = intro_m.group(1).strip().title()
            continue

        if current_role and stripped:
            if current_role == "attorney" and not current_attorney:
                intro_m = intro_pattern.search(stripped)
                if intro_m:
                    current_attorney = intro_m.group(1).strip().title()
            current_text_lines.append(stripped)

    if current_role and current_text_lines:
        flush(current_role, current_attorney, current_text_lines)

    for i, t in enumerate(turns):
        t["turn_index"] = i

    if all(t["attorney"] is None for t in turns if t["role"] == "attorney"):
        for t in turns:
            if t["role"] == "attorney":
                m = intro_pattern.search(t["text"])
                if m:
                    name = m.group(1).strip().title()
                    for tt in turns:
                        if tt["attorney"] is None:
                            tt["attorney"] = name
                    break

    return turns


def get_main_attorney(turns: list[dict]) -> str:
    """Return the attorney name with the most Q turns."""
    attorney_counts = Counter(
        t["attorney"] for t in turns
        if t["role"] == "attorney" and t["attorney"]
    )
    if not attorney_counts:
        return None
    return attorney_counts.most_common(1)[0][0]


# Quick test
test_turns = parse_transcript(
    f"{TRANSCRIPTS_DIR}/alvarado_tracie_09_11_23_full.txt"
)
main_atty  = get_main_attorney(test_turns)
atty_turns = [t for t in test_turns if t["attorney"] == main_atty and t["role"] == "attorney"]
witn_turns = [t for t in test_turns if t["role"] == "witness"]

print(f"Main attorney  : {main_atty}")
print(f"Attorney turns : {len(atty_turns)}")
print(f"Witness turns  : {len(witn_turns)}")
if atty_turns:
    print(f"Sample Q: {atty_turns[2]['text'][:120]}")
if witn_turns:
    print(f"Sample A: {witn_turns[2]['text'][:120]}")

## Encode and Project


In [ ]:
def mean_pool_from(hidden, start):
    sliced = hidden[start:]
    if sliced.shape[0] == 0:
        sliced = hidden
    return sliced.mean(dim=0)

@torch.inference_mode()
def encode_texts(texts, layer, token_start, batch_size):
    all_vecs = []
    for i in range(0, len(texts), batch_size):
        batch = texts[i : i + batch_size]
        enc = tokenizer(batch, return_tensors="pt", padding=True,
                        truncation=True, max_length=512).to(device)
        out = model(**enc, output_hidden_states=True, use_cache=False)
        layer_h = out.hidden_states[layer].float().cpu()
        mask    = enc["attention_mask"].cpu()
        for b in range(layer_h.shape[0]):
            seq_len = mask[b].sum().item()
            h   = layer_h[b, :seq_len, :]
            vec = mean_pool_from(h, token_start)
            all_vecs.append(vec)
    return torch.stack(all_vecs, dim=0)

def score_turns(turns: list[dict]) -> np.ndarray:
    """Returns (N, E) cosine similarity matrix for a list of turns."""
    texts = [t["text"] for t in turns]
    acts  = encode_texts(texts, TARGET_LAYER, TOKEN_START, BATCH_SIZE)
    acts  = F.normalize(acts.float(), dim=1)
    scores = (acts @ mat.T).numpy()  # (N, E)
    return scores


def process_deposition(path: str) -> dict | None:
    """Parse and encode one deposition. Returns result dict or None on failure."""
    name  = Path(path).stem
    turns = parse_transcript(path)
    if not turns:
        print(f"  SKIP {name} — no turns parsed")
        return None

    main_atty  = get_main_attorney(turns)
    atty_turns = [t for t in turns if t["attorney"] == main_atty and t["role"] == "attorney"]
    witn_turns = [t for t in turns if t["role"] == "witness"]

    if len(atty_turns) < 5 or len(witn_turns) < 5:
        print(f"  SKIP {name} — too few turns (atty={len(atty_turns)}, witness={len(witn_turns)})")
        return None

    atty_scores = score_turns(atty_turns)  # (Q, E)
    witn_scores = score_turns(witn_turns)  # (A, E)

    return {
        "name":        name,
        "attorney":    main_atty,
        "atty_scores": atty_scores,
        "witn_scores": witn_scores,
        "emotions":    emotions,
    }

print("Functions defined. Ready to process depositions.")


## Process All Depositions


In [ ]:
transcript_paths = sorted(glob.glob(f"{TRANSCRIPTS_DIR}/*.txt"))
# Deduplicate — skip errata/condensed duplicates
transcript_paths = [
    p for p in transcript_paths
    if not any(x in p for x in ["errata", "condensed", "links_to_ex", "_1.txt"])
]
print(f"Found {len(transcript_paths)} transcripts to process\n")

all_results = {}

for path in transcript_paths:
    name      = Path(path).stem
    cache     = RESULTS_DIR / f"{name}.npz"

    # Load from cache if available
    if cache.exists():
        data = np.load(cache, allow_pickle=True)
        all_results[name] = {
            "name":        name,
            "attorney":    str(data["attorney"]),
            "atty_scores": data["atty_scores"],
            "witn_scores": data["witn_scores"],
            "emotions":    list(data["emotions"]),
        }
        print(f"  [cached] {name}")
        continue

    print(f"  Processing: {name}")
    result = process_deposition(path)
    if result is None:
        continue

    # Cache to disk
    np.savez(cache,
        attorney    = result["attorney"],
        atty_scores = result["atty_scores"],
        witn_scores = result["witn_scores"],
        emotions    = result["emotions"],
    )
    all_results[name] = result

print(f"\nDone. {len(all_results)} depositions processed.")
